In [3]:
!pip -q install faiss-cpu sentence-transformers transformers

In [5]:
# ==============================
# Simple RAG Chatbot (Hugging Face - FREE)
# ==============================

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ==============================
# 📚 Sample Knowledge Base
# ==============================
documents = [
    "Machine learning is a field of artificial intelligence that enables systems to learn from data.",
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "Natural Language Processing (NLP) helps computers understand human language.",
    "Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.",
    "FAISS is a library developed by Facebook AI for efficient similarity search."
]

# ==============================
# 🔎 Create Embeddings
# ==============================
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(documents)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# ==============================
# 🤖 Load Hugging Face Model
# ==============================
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

# ==============================
# 💬 RAG Chat Function
# ==============================
def rag_chat(query, k=2):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    retrieved_docs = "\n".join([documents[i] for i in indices[0]])

    prompt = f"""
Context:
{retrieved_docs}

Question:
{query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=100,   # ✅ FIXED
        num_return_sequences=1
    )

    answer = response[0]['generated_text']

    return answer.split("Answer:")[-1].strip()

# ==============================
# 🧠 Chat Loop
# ==============================
print("🤖 RAG Chatbot Ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    answer = rag_chat(user_input)
    print("Bot:", answer)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🤖 RAG Chatbot Ready! Type 'exit' to stop.

You: what is machine learning


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Is it a system that learns from data

You: exit
Bot: Goodbye!
